In [1]:
import os
import re
import string
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("✅ Libraries imported successfully.")

✅ Libraries imported successfully.


In [2]:
data_path = r'C:\EDP-workspace\Fake-News-Detector\backend\data\cleaned_news_dataset.csv'
df = pd.read_csv(data_path)
df['text'] = df['text'].fillna('')

def advanced_clean_text(text):
    text = text.lower()
    # Strip publisher leakages (e.g., 'reuters', 'washington (reuters)')
    text = re.sub(r'\(?reuters\)?', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(f'[{re.escape(string.punctuation)}]', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("⏳ Cleaning text and removing publisher leakage...")
df['clean_text'] = df['text'].apply(advanced_clean_text)
print(f"✅ Loaded and cleaned {len(df)} records.")

⏳ Cleaning text and removing publisher leakage...
✅ Loaded and cleaned 44689 records.


In [3]:
X = df['clean_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"✅ Train samples: {len(X_train)} | Test samples: {len(X_test)}")

✅ Train samples: 35751 | Test samples: 8938


In [4]:
print("⏳ Extracting features using TF-IDF (Unigrams + Bigrams)...")
tfidf = TfidfVectorizer(max_features=10000, stop_words='english', ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)
print(f"✅ TF-IDF Matrix Shape: {X_train_tfidf.shape}")

⏳ Extracting features using TF-IDF (Unigrams + Bigrams)...
✅ TF-IDF Matrix Shape: (35751, 10000)


In [5]:
print("--- 1. Multinomial Naive Bayes ---")
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
y_pred_nb = nb_model.predict(X_test_tfidf)

print(f"Naive Bayes Accuracy: {accuracy_score(y_test, y_pred_nb) * 100:.2f}%\n")
print(classification_report(y_test, y_pred_nb, target_names=['REAL (0)', 'FAKE (1)']))

--- 1. Multinomial Naive Bayes ---
Naive Bayes Accuracy: 94.03%

              precision    recall  f1-score   support

    REAL (0)       0.94      0.94      0.94      4242
    FAKE (1)       0.94      0.94      0.94      4696

    accuracy                           0.94      8938
   macro avg       0.94      0.94      0.94      8938
weighted avg       0.94      0.94      0.94      8938



In [6]:
print("--- 2. Logistic Regression (TF-IDF) ---")
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)
y_pred_lr = lr_model.predict(X_test_tfidf)

print(f"Logistic Regression Accuracy: {accuracy_score(y_test, y_pred_lr) * 100:.2f}%\n")
print(classification_report(y_test, y_pred_lr, target_names=['REAL (0)', 'FAKE (1)']))

--- 2. Logistic Regression (TF-IDF) ---
Logistic Regression Accuracy: 97.94%

              precision    recall  f1-score   support

    REAL (0)       0.97      0.98      0.98      4242
    FAKE (1)       0.98      0.98      0.98      4696

    accuracy                           0.98      8938
   macro avg       0.98      0.98      0.98      8938
weighted avg       0.98      0.98      0.98      8938



In [7]:
model_dir = r'C:\EDP-workspace\Fake-News-Detector\backend\models'
os.makedirs(model_dir, exist_ok=True)

joblib.dump(lr_model, os.path.join(model_dir, "fake_news_model.pkl"))
joblib.dump(tfidf, os.path.join(model_dir, "vectorizer.pkl"))
print("✅ Saved new TF-IDF vectorizer and model binaries!")

✅ Saved new TF-IDF vectorizer and model binaries!
